In [1]:
!pip install ipython-sql jupysql

In [2]:
import sqlite3
import pandas as pd

# Load the SQL magic extension built into Jupyter
%load_ext sql

# Connect the notebook directly to the database file
%sql sqlite:///priorauth_ops.db

Connecting to 'sqlite:///priorauth_ops.db'

In [3]:
# Query A: Identify the top drugs that have a
# Prior Authorization (PA) denial rate higher than 20%

In [4]:
%%sql
SELECT
    Drug_name,
    COUNT(Claim_ID) AS Total_Claims,
    SUM(CASE WHEN Status = 'Denied' THEN 1 ELSE 0 END) AS Total_Denials,
    ROUND(
        (SUM(CASE WHEN Status = 'Denied' THEN 1 ELSE 0 END) * 100.0) / COUNT(Claim_ID),
            2
    ) AS Denial_Rate_Percentage
FROM pharmacy_claims
GROUP BY Drug_Name
HAVING Denial_Rate_Percentage > 20.0
ORDER BY Denial_Rate_Percentage DESC;

Running query in 'sqlite:///priorauth_ops.db'

Drug_Name,Total_Claims,Total_Denials,Denial_Rate_Percentage
Ozempic,1493,376,25.18
Keytruda,1377,315,22.88
Humira,1452,326,22.45


In [5]:
# Query A Explanation
# If a manager asks why a certain drug is frustrating patients,
# this query gives you the exact data-backed answer

# SUM(CASE WHEN): Acts like an intelligent counter. It goes
# row by row through each record, counting a 1 every time it sees
# "Denied" and a 0 for everything else

# GROUP BY Drug_Name: Instead of looking at a massive wall of
# text, it squashes all the rows down into a neat summary
# bucket for each individual drug (e.g., Humira, Ozempic, Metformin)

# HAVING... > 20.0: This acts as an automated filter, hiding the
# "good" drugs and immediately exposing the problem areas that 
# require manager intervention.

In [6]:
# Query B: Calculate the  total "abandoned revenue" (claims that were rejected 
# for PA and never re-submitted/filled).

# Explanation: This query targets patients or providers who got hit 
# with a Prior Authorization hurdle and completely walked away from 
# the medication ('Status' = Abandoned). For a manager, this highlights 
# lost business and potential gaps in patient care. 

In [7]:
%%sql
SELECT
    Drug_Class,
    COUNT(Claim_ID) AS Abandoned_Claims_Count,
    -- We use a standard estimated cost for abandoned specialty/maintenance drugs to show impact
    SUM(CASE WHEN Drug_Class = 'Specialty' THEN 1200.00 ELSE 45.00 END) 
    AS Estimated_Abandoned_Revenue_Loss

FROM pharmacy_claims
WHERE Status = 'Abandoned'
GROUP BY Drug_Class;

Running query in 'sqlite:///priorauth_ops.db'

Drug_Class,Abandoned_Claims_Count,Estimated_Abandoned_Revenue_Loss
Specialty,998,1197600.0


In [8]:
# The query B finding means that the pharmacy lost $1,197,600. Looking at the count means
# that 998 patients walked away without receiving their medication. This happens because
# specialty drugs are so expensive, which results in patients often getting overwhelmed by 
# paperwork or seeing the "sticker price" of the medication when an insurance rejection
# hits the pharmacy counter.

In [9]:
# Query C: Segment approval rates by "Plan Type" to see which insurance plans are the most restrictive.

# Explanation: This helps a PBM analyst see exactly where the tightest restrictions are. 
# You'll see the exact percentage of claims that slide through smoothly versus those that 
# require heavy clinical review.

In [10]:
%%sql
SELECT 
    Drug_Class,
    COUNT(Claim_ID) AS Total_Submissions,
    SUM(CASE WHEN Status = 'Approved' THEN 1 ELSE 0 END) AS Total_Approvals,
    ROUND(
        (SUM(CASE WHEN Status = 'Approved' THEN 1 ELSE 0 END) * 100.0) / COUNT(Claim_ID), 
        2
    ) AS Approval_Rate_Percentage

FROM pharmacy_claims
GROUP BY Drug_Class
ORDER BY Approval_Rate_Percentage ASC;

Running query in 'sqlite:///priorauth_ops.db'

Drug_Class,Total_Submissions,Total_Approvals,Approval_Rate_Percentage
Specialty,4322,2307,53.38
Acute,1438,1438,100.0
Maintenance,4240,4240,100.0


In [11]:
# Query C Findings

# Acute (100% approval / 1438 submissions): Exactly how a smooth pharmacy
# operates. Acute drugs (like antibiotics) need to be filled immediately
# so a patient can start healing today.

# Maintenance (100% approval / 4240 submissions): Standard daily medications
# (like blood pressure or cholesterol pills) are slipping through without issue,
# keeping chronic conditions stable

# Specialty (53.38% approval / 4322 submissions): The prior authorization process is
# heavily limiting access to these high-cost drugs

In [12]:
import sqlite3
import pandas as pd

# Reconnect to the database using Python's standard driver
conn = sqlite3.connect('priorauth_ops.db')

# 1. Pull the massive raw claims table to a DataFrame and export it
df_raw = pd.read_sql_query("SELECT * FROM pharmacy_claims;", conn)
df_raw.to_csv('raw_pharmacy_claims_10k.csv', index=False)

# 2. Pull your high-denial drug summary and export it
df_denials = pd.read_sql_query("""
    SELECT Drug_Name, COUNT(Claim_ID) AS Total_Claims, SUM(CASE WHEN Status = 'Denied' THEN 1 ELSE 0 END) AS Total_Denials,
           ROUND((SUM(CASE WHEN Status = 'Denied' THEN 1 ELSE 0 END) * 100.0) / COUNT(Claim_ID), 2) AS Denial_Rate_Percentage
    FROM pharmacy_claims GROUP BY Drug_Name HAVING Denial_Rate_Percentage > 20.0 ORDER BY Denial_Rate_Percentage DESC;
""", conn)
df_denials.to_csv('high_denial_drugs_summary.csv', index=False)

conn.close()
print("📁 Success! Your CSV export files are ready in your project directory!")

📁 Success! Your CSV export files are ready in your project directory!


In [13]:
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

# Create the workbook
wb = Workbook()

# Set up a professional, clean corporate palette (Classic Navy & Slate)
navy_dark = "1B365D"
zebra_light = "F4F7FA"
text_white = "FFFFFF"
border_gray = "D3D3D3"

font_title = Font(name="Calibri", size=16, bold=True, color=navy_dark)
font_subtitle = Font(name="Calibri", size=11, italic=True, color="555555")
font_header = Font(name="Calibri", size=11, bold=True, color=text_white)
font_data = Font(name="Calibri", size=11, color="000000")
font_total = Font(name="Calibri", size=11, bold=True, color="000000")

fill_header = PatternFill(start_color=navy_dark, end_color=navy_dark, fill_type="solid")
fill_zebra = PatternFill(start_color=zebra_light, end_color=zebra_light, fill_type="solid")

thin_side = Side(border_style="thin", color=border_gray)
border_data = Border(left=thin_side, right=thin_side, top=thin_side, bottom=thin_side)
border_total = Border(top=Side(border_style="thin", color="000000"), bottom=Side(border_style="double", color="000000"))

# --- SHEET 1: EXECUTIVE SUMMARY ---
ws1 = wb.active
ws1.title = "Executive Summary"
ws1.views.sheetView[0].showGridLines = True

ws1["B2"] = "Prior Authorization Operational Performance Report"
ws1["B2"].font = font_title
ws1["B3"] = "Enterprise Scale Executive Performance Analytics (10,000 Baseline Simulation)"
ws1["B3"].font = font_subtitle

# Layout KPI Blocks (Matching your 10k run exactly!)
ws1["B5"] = "Total Revenue Leakage"
ws1["B5"].font = Font(name="Calibri", size=10, bold=True, color="555555")
ws1["B6"] = 1197600
ws1["B6"].font = Font(name="Calibri", size=20, bold=True, color="C00000")
ws1["B6"].number_format = "$#,##0"

ws1["E5"] = "Abandoned Specialty Claims"
ws1["E5"].font = Font(name="Calibri", size=10, bold=True, color="555555")
ws1["E6"] = 998
ws1["E6"].font = Font(name="Calibri", size=20, bold=True, color="000000")
ws1["E6"].number_format = "#,##0"

ws1["H5"] = "Specialty Approval Rate"
ws1["H5"].font = Font(name="Calibri", size=10, bold=True, color="555555")
ws1["H6"] = 0.5338
ws1["H6"].font = Font(name="Calibri", size=20, bold=True, color="4A90E2")
ws1["H6"].number_format = "0.00%"


# --- SHEET 2: DRUG CLASS SUMMARY ---
ws2 = wb.create_sheet(title="Drug Class Summary")
ws2.views.sheetView[0].showGridLines = True
ws2["B2"] = "Operational Summary by Drug Class"
ws2["B2"].font = font_title

headers_class = ["Drug Class", "Total Submissions", "Total Approvals", "Approval Rate"]
for col_idx, text in enumerate(headers_class, start=2):
    cell = ws2.cell(row=4, column=col_idx, value=text)
    cell.font = font_header
    cell.fill = fill_header
    cell.alignment = Alignment(horizontal="center")

class_data = [
    ["Specialty", 4322, 2307, "=D5/C5"],
    ["Acute", 1438, 1438, "=D6/C6"],
    ["Maintenance", 4240, 4240, "=D7/C7"]
]

for row_offset, row_data in enumerate(class_data, start=5):
    for col_offset, val in enumerate(row_data, start=2):
        cell = ws2.cell(row=row_offset, column=col_offset, value=val)
        cell.font = font_data
        cell.border = border_data
        if col_offset == 2:
            cell.alignment = Alignment(horizontal="left")
        elif col_offset in [3, 4]:
            cell.alignment = Alignment(horizontal="right")
            cell.number_format = "#,##0"
        elif col_offset == 5:
            cell.alignment = Alignment(horizontal="right")
            cell.number_format = "0.00%"
        if row_offset % 2 == 1:
            cell.fill = fill_zebra

# Add dynamic total formulas
ws2["B8"] = "Total / Average"
ws2["B8"].font = font_total
ws2["B8"].border = border_total
ws2["C8"] = "=SUM(C5:C7)"
ws2["C8"].font = font_total
ws2["C8"].number_format = "#,##0"
ws2["C8"].border = border_total
ws2["D8"] = "=SUM(D5:D7)"
ws2["D8"].font = font_total
ws2["D8"].number_format = "#,##0"
ws2["D8"].border = border_total
ws2["E8"] = "=D8/C8"
ws2["E8"].font = font_total
ws2["E8"].number_format = "0.00%"
ws2["E8"].border = border_total


# --- SHEET 3: TOP DENIAL ANALYTICS ---
ws3 = wb.create_sheet(title="Top Denial Analytics")
ws3.views.sheetView[0].showGridLines = True
ws3["B2"] = "Prior Authorization Denial Exceptions (>20% Threshold)"
ws3["B2"].font = font_title

headers_drug = ["Drug Name", "Total Claims", "Total Denials", "Denial Rate"]
for col_idx, text in enumerate(headers_drug, start=2):
    cell = ws3.cell(row=4, column=col_idx, value=text)
    cell.font = font_header
    cell.fill = fill_header
    cell.alignment = Alignment(horizontal="center")

drug_data = [
    ["Ozempic", 1493, 376, "=D5/C5"],
    ["Keytruda", 1377, 315, "=D6/C6"],
    ["Humira", 1452, 326, "=D7/C7"]
]

for row_offset, row_data in enumerate(drug_data, start=5):
    for col_offset, val in enumerate(row_data, start=2):
        cell = ws3.cell(row=row_offset, column=col_offset, value=val)
        cell.font = font_data
        cell.border = border_data
        if col_offset == 2:
            cell.alignment = Alignment(horizontal="left")
        elif col_offset in [3, 4]:
            cell.alignment = Alignment(horizontal="right")
            cell.number_format = "#,##0"
        elif col_offset == 5:
            cell.alignment = Alignment(horizontal="right")
            cell.number_format = "0.00%"
        if row_offset % 2 == 1:
            cell.fill = fill_zebra

# Enforce clean margins and layout spacing dynamically
for ws in [ws1, ws2, ws3]:
    ws.column_dimensions['A'].width = 3
    ws.column_dimensions['B'].width = 28
    ws.column_dimensions['C'].width = 20
    ws.column_dimensions['D'].width = 20
    ws.column_dimensions['E'].width = 20

# Save out directly into your project workspace
wb.save("PA_Operational_Executive_Report.xlsx")
print("📁 Success! 'PA_Operational_Executive_Report.xlsx' has been created directly in your folder!")

📁 Success! 'PA_Operational_Executive_Report.xlsx' has been created directly in your folder!
